# Heston Model Calibration with Real Market Data

This notebook demonstrates Heston model calibration using real market data from yfinance with different objective functions.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft
from scipy.integrate import quad
from scipy.optimize import minimize
from scipy.stats import norm
from typing import Dict, Tuple, Optional
from datetime import datetime, timedelta

plt.style.use('seaborn-v0_8')
np.set_printoptions(precision=4, suppress=True)

## 2. Heston Model Implementation

In [ ]:
class HestonModel:
    """
    Heston stochastic volatility model for option pricing.
    
    Parameters:
    -----------
    S0 : float
        Initial stock price
    r : float
        Risk-free rate
    q : float
        Dividend yield
    theta : np.ndarray
        Heston parameters: [nu0, theta, rho, kappa, sigma]
        - nu0: initial variance
        - theta: long-term variance
        - rho: correlation between asset and variance
        - kappa: mean-reversion speed
        - sigma: volatility of variance
    """
    
    def __init__(self, S0: float, r: float, q: float, theta: np.ndarray):
        self.S0 = S0
        self.r = r
        self.q = q
        self.theta = theta
        self.nu0, self.theta_v, self.rho, self.kappa, self.sigma = theta
    
    def characteristic_function(self, phi: np.ndarray, tau: float) -> np.ndarray:
        """
        Heston characteristic function.
        """
        nu0, theta_v, rho, kappa, sigma = self.theta
        
        d = np.sqrt((rho * sigma * 1j * phi - kappa)**2 + 
                    sigma**2 * (1j * phi + phi**2))
        g = (kappa - rho * sigma * 1j * phi - d) / (kappa - rho * sigma * 1j * phi + d)
        
        C = (self.r - self.q) * 1j * phi * tau + \
            (kappa * theta_v / sigma**2) * ((kappa - rho * sigma * 1j * phi - d) * tau - 
                                            2 * np.log((1 - g * np.exp(-d * tau)) / (1 - g)))
        D = ((kappa - rho * sigma * 1j * phi - d) / sigma**2) * \
            (1 - np.exp(-d * tau)) / (1 - g * np.exp(-d * tau))
        
        return np.exp(C + D * nu0 + 1j * phi * np.log(self.S0))
    
    def call_price_fft(self, K: np.ndarray, tau: float, 
                       alpha: float = 1.5, n: int = 4096, 
                       B: float = 200.0) -> np.ndarray:
        """
        Compute call prices using FFT method.
        """
        eta = 2 * B / n
        lambda_ = 2 * np.pi / (n * eta)
        
        v = np.arange(n) * eta
        k = np.arange(n) * lambda_ - B + lambda_ / 2
        
        psi = np.exp(-self.r * tau) * \
              (self.characteristic_function(v - (alpha + 1) * 1j, tau) / 
               ((alpha + 1j * v) * (alpha + 1 + 1j * v)))
        
        fft_values = fft(psi) * eta
        call_prices = np.exp(-alpha * k) / np.pi * np.real(fft_values)
        
        from scipy.interpolate import interp1d
        log_K = np.log(K)
        log_k = np.log(self.S0) + k
        
        log_K = np.clip(log_K, log_k[0], log_k[-1])
        
        interp_func = interp1d(log_k, call_prices, kind='cubic', 
                              fill_value='extrapolate')
        prices = interp_func(log_K)
        prices = np.maximum(prices, 0)
        
        return prices
    
    def implied_volatility(self, price: float, K: float, tau: float, 
                          max_iter: int = 100, tol: float = 1e-8) -> float:
        """
        Compute implied volatility from option price using Newton-Raphson.
        """
        def black_scholes_call(S, K, tau, r, q, sigma):
            d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
            d2 = d1 - sigma * np.sqrt(tau)
            return S * np.exp(-q * tau) * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
        
        def vega(S, K, tau, r, q, sigma):
            d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
            return S * np.exp(-q * tau) * norm.pdf(d1) * np.sqrt(tau)
        
        sigma = 0.3
        
        for _ in range(max_iter):
            bs_price = black_scholes_call(self.S0, K, tau, self.r, self.q, sigma)
            diff = bs_price - price
            
            if abs(diff) < tol:
                break
            
            v = vega(self.S0, K, tau, self.r, self.q, sigma)
            
            if v < 1e-10:
                break
            
            sigma = sigma - diff / v
            sigma = max(sigma, 0.01)
        
        return sigma

## 3. Objective Functions

In [ ]:
def objective_sse(theta: np.ndarray, market_data: dict, model: HestonModel) -> float:
    """
    Objective function 1: Sum of squared errors in option prices.
    
    G(Θ) = Σ (C_model(Θ) - C_market)^2
    """
    total_error = 0.0
    
    for i in range(len(market_data['K'])):
        K = market_data['K'][i]
        tau = market_data['T'][i]
        market_price = market_data['price'][i]
        
        model.theta = theta
        model.nu0, model.theta_v, model.rho, model.kappa, model.sigma = theta
        
        model_price = model.call_price_fft(np.array([K]), tau)[0]
        total_error += (model_price - market_price)**2
    
    return total_error

def objective_weighted_sse_vega(theta: np.ndarray, market_data: dict, 
                                 model: HestonModel) -> float:
    """
    Objective function 2a: Weighted sum of squared errors with vega weights.
    
    G(Θ) = Σ w_i * (C_model(Θ) - C_market)^2
    
    where w_i = 1/vega
    """
    total_error = 0.0
    
    for i in range(len(market_data['K'])):
        K = market_data['K'][i]
        tau = market_data['T'][i]
        market_price = market_data['price'][i]
        
        model.theta = theta
        model.nu0, model.theta_v, model.rho, model.kappa, model.sigma = theta
        
        model_price = model.call_price_fft(np.array([K]), tau)[0]
        
        # Weight by 1/vega (using market price as proxy for vega)
        weight = 1.0 / (market_price + 1e-10)
        
        total_error += weight * (model_price - market_price)**2
    
    return total_error

def objective_weighted_sse_spread(theta: np.ndarray, market_data: dict, 
                                  model: HestonModel) -> float:
    """
    Objective function 2b: Weighted sum of squared errors with spread weights.
    
    G(Θ) = Σ w_i * (C_model(Θ) - C_market)^2
    
    where w_i = 1/Spread
    """
    total_error = 0.0
    
    for i in range(len(market_data['K'])):
        K = market_data['K'][i]
        tau = market_data['T'][i]
        market_price = market_data['price'][i]
        
        model.theta = theta
        model.nu0, model.theta_v, model.rho, model.kappa, model.sigma = theta
        
        model_price = model.call_price_fft(np.array([K]), tau)[0]
        
        # Weight by 1/spread
        if 'spread' in market_data:
            weight = 1.0 / (market_data['spread'][i] + 1e-10)
        else:
            weight = 1.0 / (market_price + 1e-10)
        
        total_error += weight * (model_price - market_price)**2
    
    return total_error

def objective_relative_sse(theta: np.ndarray, market_data: dict, 
                           model: HestonModel) -> float:
    """
    Objective function 3: Relative squared errors.
    
    G(Θ) = Σ ((C_model(Θ) - C_market) / C_market)^2
    """
    total_error = 0.0
    
    for i in range(len(market_data['K'])):
        K = market_data['K'][i]
        tau = market_data['T'][i]
        market_price = market_data['price'][i]
        
        model.theta = theta
        model.nu0, model.theta_v, model.rho, model.kappa, model.sigma = theta
        
        model_price = model.call_price_fft(np.array([K]), tau)[0]
        
        if market_price > 1e-10:
            total_error += ((model_price - market_price) / market_price)**2
    
    return total_error

def objective_iv_sse(theta: np.ndarray, market_data: dict, 
                      model: HestonModel) -> float:
    """
    Objective function 4: Sum of squared errors in implied volatilities.
    
    G(Θ) = Σ (IV_model(Θ) - IV_market)^2
    """
    total_error = 0.0
    
    for i in range(len(market_data['K'])):
        K = market_data['K'][i]
        tau = market_data['T'][i]
        market_price = market_data['price'][i]
        
        model.theta = theta
        model.nu0, model.theta_v, model.rho, model.kappa, model.sigma = theta
        
        model_price = model.call_price_fft(np.array([K]), tau)[0]
        model_iv = model.implied_volatility(model_price, K, tau)
        
        market_iv = model.implied_volatility(market_price, K, tau)
        
        total_error += (model_iv - market_iv)**2
    
    return total_error

## 4. Fetch Real Market Data from yfinance

In [ ]:
def fetch_option_data(ticker: str, expiry_date: str = None) -> dict:
    """
    Fetch option chain data from yfinance.
    
    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g., 'AAPL')
    expiry_date : str, optional
        Option expiry date in 'YYYY-MM-DD' format
    
    Returns:
    --------
    dict
        Dictionary containing market data
    """
    stock = yf.Ticker(ticker)
    
    # Get current stock price
    S0 = stock.history(period='1d')['Close'].iloc[-1]
    print(f"Current stock price: ${S0:.2f}")
    
    # Get available expiry dates
    expirations = stock.options
    print(f"Available expiry dates: {expirations}")
    
    if expiry_date is None:
        # Use the nearest expiry
        expiry_date = expirations[0]
    print(f"Using expiry date: {expiry_date}")
    
    # Get option chain
    opt = stock.option_chain(expiry_date)
    
    # Get call options
    calls = opt.calls
    
    # Filter for liquid options (reasonable volume and open interest)
    calls = calls[(calls['volume'] > 10) & (calls['openInterest'] > 10)]
    
    # Calculate time to maturity
    expiry = datetime.strptime(expiry_date, '%Y-%m-%d')
    today = datetime.now()
    T = (expiry - today).days / 365.0
    
    # Prepare market data
    market_data = {
        'K': calls['strike'].values,
        'T': np.full(len(calls), T),
        'price': calls['lastPrice'].values,
        'spread': (calls['ask'] - calls['bid']).values,
        'impliedVol': calls['impliedVolatility'].values
    }
    
    # Filter out invalid data
    valid_mask = (market_data['price'] > 0) & (market_data['impliedVol'] > 0)
    for key in market_data:
        market_data[key] = market_data[key][valid_mask]
    
    print(f"Number of valid options: {len(market_data['K'])}")
    print(f"Time to maturity: {T:.3f} years")
    
    return market_data, S0

## 5. Calibration Function

In [ ]:
def calibrate_heston(market_data: dict, S0: float, r: float, q: float,
                     objective_func, initial_guess: np.ndarray = None,
                     method: str = 'SLSQP') -> dict:
    """
    Calibrate Heston model parameters.
    
    Parameters:
    -----------
    market_data : dict
        Market data dictionary
    S0 : float
        Current stock price
    r : float
        Risk-free rate
    q : float
        Dividend yield
    objective_func : callable
        Objective function to minimize
    initial_guess : np.ndarray, optional
        Initial parameter guess
    method : str
        Optimization method
    
    Returns:
    --------
    dict
        Calibration results
    """
    # Default initial guess
    if initial_guess is None:
        initial_guess = np.array([0.04, 0.04, -0.5, 2.0, 0.3])
    
    # Parameter bounds
    bounds = [
        (0.001, 1.0),    # nu0: initial variance
        (0.001, 1.0),    # theta: long-term variance
        (-0.999, 0.999), # rho: correlation
        (0.001, 10.0),   # kappa: mean-reversion speed
        (0.001, 2.0)     # sigma: volatility of variance
    ]
    
    # Create model
    model = HestonModel(S0, r, q, initial_guess)
    
    # Define objective wrapper
    def objective_wrapper(theta):
        return objective_func(theta, market_data, model)
    
    # Optimize
    result = minimize(
        objective_wrapper,
        initial_guess,
        method=method,
        bounds=bounds,
        options={'maxiter': 1000, 'ftol': 1e-8}
    )
    
    return {
        'theta': result.x,
        'objective_value': result.fun,
        'success': result.success,
        'message': result.message,
        'model': model
    }

## 6. Fetch Data and Calibrate with All Objective Functions

In [ ]:
# Set parameters
TICKER = 'AAPL'  # Change to your desired ticker
R = 0.05  # Risk-free rate
Q = 0.0   # Dividend yield

# Fetch market data
market_data, S0 = fetch_option_data(TICKER)

# Define objective functions
objective_functions = {
    'SSE': objective_sse,
    'Weighted SSE (Vega)': objective_weighted_sse_vega,
    'Weighted SSE (Spread)': objective_weighted_sse_spread,
    'Relative SSE': objective_relative_sse,
    'IV SSE': objective_iv_sse
}

## 7. Run Calibration with Each Objective Function

In [ ]:
results = {}

print("="*80)
print("Heston Model Calibration with Different Objective Functions")
print("="*80)
print()

for name, obj_func in objective_functions.items():
    print(f"Calibrating with {name}...")
    
    result = calibrate_heston(market_data, S0, R, Q, obj_func, method='SLSQP')
    results[name] = result
    
    theta = result['theta']
    print(f"  Parameters: nu0={theta[0]:.4f}, theta={theta[1]:.4f}, "
          f"rho={theta[2]:.4f}, kappa={theta[3]:.4f}, sigma={theta[4]:.4f}")
    print(f"  Objective value: {result['objective_value']:.6f}")
    print(f"  Success: {result['success']}")
    print()